In [1]:
import sys
print(sys.executable)

C:\Users\Kaushik Das\anaconda3\envs\nlp_env\python.exe


In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os
from tqdm import tqdm

In [4]:
df = pd.read_csv("shl_assessments.csv")

print(f"Total assessments loaded: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample data:")
df.head(5)

Total assessments loaded: 377

Columns: ['name', 'url', 'remote_testing', 'adaptive_irt', 'test_type', 'description']

Sample data:


,name,url,remote_testing,adaptive_irt,test_type,description
0,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,Yes,No,A\nE\nB\nC\nD\nP,Description\nThis report is designed to be giv...
1,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,Yes,Yes,K,Description\nThe.NET Framework 4.5 test measur...
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,Yes,No,K,Description\nMulti-choice test that measures t...
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,Yes,No,K,Description\nMulti-choice test that measures t...
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,Yes,No,K,Description\nMulti-choice test that measures t...


In [7]:
def clean_data(df):
    
    df["description"] = df["description"].fillna("")
    df["test_type"] = df["test_type"].fillna("")
    df["remote_testing"] = df["remote_testing"].fillna("No")
    df["adaptive_irt"] = df["adaptive_irt"].fillna("No")
    df["name"] = df["name"].fillna("")
    
    
    df = df[df["name"] != ""]    
    df = df.drop_duplicates(subset=["url"])
    df = df.reset_index(drop=True)
    
    print(f"✅ Data cleaned!")
    print(f"Total assessments after cleaning: {len(df)}")
    return df

df = clean_data(df)

✅ Data cleaned!
Total assessments after cleaning: 377


In [12]:
def create_combined_text(row):
    test_type_map = {
        "A": "Ability and Aptitude",
        "B": "Biodata and Situational Judgement",
        "C": "Competencies",
        "D": "Development and 360",
        "E": "Assessment Exercises",
        "K": "Knowledge and Skills",
        "P": "Personality and Behavior",
        "S": "Simulations"
    }

    type_letters = str(row["test_type"]).split()
    type_full = " ".join([test_type_map.get(t, t) for t in type_letters])

    text = f"""
    {row['name']}
    {type_full}
    {row['description']}
    """

    return text.strip()

df["combined_text"] = df.apply(create_combined_text, axis=1)

print("\nCombined text for first assessment:")
print(df["combined_text"][0])


Combined text for first assessment:
Global Skills Development Report
    Ability and Aptitude Assessment Exercises Biodata and Situational Judgement Competencies Development and 360 Personality and Behavior
    Description
This report is designed to be given to individuals who have completed the Global Skills Assessment (GSA). With coverage across the Great 8 Domains, this measure of self-reported behaviors offers a complete overview of their current skills. Participants receive actionable tips on leveraging their top skill strengths and how they might develop their growth skills.


In [13]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"\nGenerating embeddings for {len(df)} assessments...")

texts = df["combined_text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

print(f"\n✅ Embeddings generated!")
print(f"Embedding shape: {embeddings.shape}")


Loading embedding model...


C:\Users\Kaushik Das\anaconda3\envs\nlp_env\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kaushik Das\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|█████████████████████████████████████████████████████████|


Generating embeddings for 377 assessments...


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:11<00:00,  1.08it/s]


✅ Embeddings generated!
Embedding shape: (377, 384)


In [14]:
dimension = embeddings.shape[1]
print(f"Embedding dimension..: {dimension}")

index = faiss.IndexFlatL2(dimension)

embeddings_float32 = np.array(embeddings).astype("float32")

index.add(embeddings_float32)

print(f"Total vectors in index: {index.ntotal}")

Embedding dimension..: 384
Total vectors in index: 377


In [15]:
os.makedirs("data", exist_ok=True)

df.to_csv("data/shl_assessments_clean.csv", index=False)

faiss.write_index(index, "data/shl_index.faiss")

np.save("data/embeddings.npy", embeddings_float32)

with open("data/model_name.txt", "w") as f:
    f.write("all-MiniLM-L6-v2")

print(os.listdir("data"))

['embeddings.npy', 'model_name.txt', 'shl_assessments_clean.csv', 'shl_index.faiss']
